# 08 Combining studies (meta-analysis): BCG trials, passive smoking, stroke units, CO2 and plants, warfarin, interviews

`table.combine_studies` (added after simplydata 0.2.0) on six published meta-analysis datasets shipped with the R
package metadat:

- **BCG vaccine trials** (Colditz et al. 1994), the worked example of Viechtbauer (2010), *Conducting meta-analyses
  in R with the metafor package*, JSS 36(3), whose printed values are the published reference: risk ratios from
  counts with every `method=` (REML, DerSimonian-Laird, Paule-Mandel, a common effect, Mantel-Haenszel, Peto),
  `interval_by=`, `show=` (risk ratios, odds ratios, risk differences), subgroups (`for_each_group_in=`),
  meta-regression on latitude (`using=`), Peters' test and trim and fill;
- **passive smoking** (Hackshaw et al. 1997, BMJ): odds ratios given with their 95 % intervals (`estimates=`),
  published pooled odds ratio 1.24 (1.13 to 1.36); Egger's test and trim and fill;
- **stroke rehabilitation** (Normand 1999): averages, standardised differences (Hedges' g) and differences in days;
- **CO2 and woody plants** (Curtis & Wang 1998): the log response ratio (`show="ratios of averages"`);
- **warfarin and stroke** (Hart et al. 1999): events and patient-years (`time=`), rate ratios;
- **employment interviews** (McDaniel et al. 1994): correlations (`correlations=`, Fisher's z).

**Only simplydata runs in this notebook.** The reference values are in `references/08_combining_studies.csv`, written
by `scripts/references/08_combining_studies.py`. simplydata calls R metafor, so metafor is the original:
- **printed values** (Viechtbauer 2010; Hackshaw et al. 1997) first, within half a unit of the last printed digit;
- only where nothing is printed, **R metafor** (`escalc`, `rma`, `rma.mh`, `rma.peto`, `regtest`, `trimfill`) run once
  there, and R meta (`metabias`) for Peters' test, with the settings simplydata states it uses (REML, Hartung-Knapp
  with metafor's ad hoc rule `test = "adhoc"`, prediction interval from `predict()`). Relative difference **1e-10**
  for results from closed formulas and **1e-6** for results found by iteration; the BCG models found by iteration
  against metafor run to convergence (`tol = 1e-12`), as its defaults stop at 1e-5.

Ratios are combined on the log scale and correlations on Fisher's z scale, and are compared there: `add_column` puts
simplydata's estimates and intervals on that scale ("... on its scale"). Each check prints its source.

In [1]:
import simplydata as sdata

from checks import ReferenceChecks, data_file, quiet, read_rows, read_rows_by, versions

NOTEBOOK = "08_combining_studies"
versions("simplydata")
checks = ReferenceChecks(NOTEBOOK)
SCALES = {"same": "`{column}`", "log": "log(`{column}`)", "z": "0.5 * log((1 + `{column}`) / (1 - `{column}`))"}
INTERVAL = ["estimate", "estimate 95 % CI low", "estimate 95 % CI high"]


def load(name):
    return quiet(sdata.table, data_file(NOTEBOOK, name), separator=",")


def combine(table, scale="log", **options):
    """simplydata's answer, read row by row (table.get_row), with the estimate and its interval also on the scale
    the studies are combined on (log for ratios, Fisher's z for correlations), added with table.add_column."""
    result = quiet(table.combine_studies, **options)
    for column in INTERVAL:
        quiet(result.add_column, f"{column} on its scale", SCALES[scale].format(column=column))
    return read_rows(result)


def row_of(rows, kind, subgroup=None):
    return [row for row in rows if row["row kind"] == kind and (subgroup is None or row["subgroup"] == subgroup)][0]


def combined(rows):
    """The combined row's numbers, the estimate and interval on the scale it is combined on."""
    c = row_of(rows, "combined")
    out = {"estimate": c["estimate on its scale"], "95 % CI low": c["estimate 95 % CI low on its scale"],
           "95 % CI high": c["estimate 95 % CI high on its scale"], "standard error": c["standard error"],
           "p-value": c["p-value"], "tau squared": c["tau squared"], "I squared %": c["I squared %"], "Q": c["Q"]}
    if any(row["row kind"] == "prediction interval" for row in rows):
        p = row_of(rows, "prediction interval")
        out["prediction interval low"] = p["estimate 95 % CI low on its scale"]
        out["prediction interval high"] = p["estimate 95 % CI high on its scale"]
    return out


def each_study(dataset, method, rows, estimate_name):
    values = {}
    for row in rows:
        if row["row kind"] == "study":
            values[f"{row['study']}: {estimate_name}"] = row["estimate on its scale"]
            values[f"{row['study']}: standard error"] = row["standard error"]
    checks.check_values(dataset, method, values)

Python 3.12.16 on Linux 7.0.0-1020-azure
simplydata 0.2.0 (commit a5a8ff0 of github.com/bioforkgithub/simplydata)
simplydata 0.2.0


R version 4.6.2 (2026-10-29)


## BCG vaccine trials: risk ratios from counts

13 trials: tuberculosis cases among the vaccinated (`tpos` of `tpos + tneg`) and unvaccinated (`cpos` of
`cpos + cneg`). One full printed answer (simplydata's defaults: REML, Hartung-Knapp), then the numbers.

In [2]:
bcg = load("bcg")
BCG = dict(names="name", events=["tpos", "cpos"], out_of=["vaccinated", "unvaccinated"])
bcg.combine_studies(**BCG)

Combining 13 studies (random effects, REML, Hartung-Knapp): risk ratio of "tpos" out of "vaccinated" against "cpos" out of "unvaccinated".
Combined risk ratio 0.489 (95 % CI 0.330 to 0.726, Hartung-Knapp; 13 studies).
If there were truly no effect on average (a risk ratio of 1), a combined effect this far from 1 or further would turn up in about 2 of 1000 sets of studies like these (p = 0.002).
In a new setting like these studies, the true risk ratio would lie between 0.136 and 1.76 (95 % prediction interval, t with 12 degrees of freedom, as metafor uses with Hartung-Knapp; Higgins et al. 2009 use 11).
The true risk ratios typically differ from the average by a factor of about 1.75 up or down (tau = 0.560 on the log scale; tau-squared = 0.313, 95 % CI 0.120 to 1.11).
I-squared = 92 % (95 % CI 82 to 98 %): the share of the spread in the study results that is beyond chance; it says how much of the spread is real, not how big it is.
If every study truly measured the same effect, differenc

,study,row kind,measure,estimate,estimate 95 % CI low,estimate 95 % CI high,standard error,weight %,combined on,p-value,studies,tau,tau squared,I squared %,Q,Q degrees of freedom,Q p-value,studies added
1,Aronson 1948,study,risk ratio,0.410939,0.134302,1.257398,0.570600,5.059483,log scale (ratios),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Ferguson & Simes 1949,study,risk ratio,0.204868,0.086297,0.486352,0.441114,6.364679,log scale (ratios),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Rosenthal et al 1960,study,risk ratio,0.259740,0.073443,0.918609,0.644490,4.436027,log scale (ratios),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Hart & Sutherland 1977,study,risk ratio,0.236561,0.179281,0.312141,0.141457,9.698747,log scale (ratios),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Frimodt-Moller et al 1973,study,risk ratio,0.804490,0.516293,1.253558,0.226297,8.868456,log scale (ratios),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Stein & Aronson 1953,study,risk ratio,0.455611,0.387132,0.536203,0.083100,10.095739,log scale (ratios),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Vandiviere et al 1973,study,risk ratio,0.197721,0.078357,0.498919,0.472247,6.027181,log scale (ratios),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,TPT Madras 1980,study,risk ratio,1.012024,0.894572,1.144897,0.062941,10.189439,log scale (ratios),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Coetzee & Berjak 1968,study,risk ratio,0.625366,0.392576,0.996196,0.237559,8.743133,log scale (ratios),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,Rosenthal et al 1961,study,risk ratio,0.253765,0.149421,0.430976,0.270231,8.367607,log scale (ratios),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Published values (Viechtbauer 2010, JSS 36(3))

The random-effects model (REML) of the log risk ratios: estimate -0.7145, standard error 0.1798, 95 % CI -1.0669 to
-0.3622 (normal interval), tau-squared 0.3132, I-squared 92.22 %, Q = 152.2330 on 12 degrees of freedom; the pooled
risk ratio 0.4894 (95 % CI 0.3441 to 0.6962) with the prediction (credibility) interval 0.1546 to 1.5490. These are
printed for the normal (z) interval, so simplydata is run with `interval_by="normal"` here; its p-value, which is not
printed, is compared with R metafor. The DerSimonian-Laird tau-squared is printed as 0.3088. The mixed-effects model
with absolute latitude: intercept 0.2515, slope -0.0291.

In [3]:
normal = combine(bcg, **BCG, interval_by="normal")
ours = combined(normal)
c, p = row_of(normal, "combined"), row_of(normal, "prediction interval")
checks.check_values("BCG trials", "combine_studies (REML, normal)", {
    "log risk ratio": ours["estimate"], "standard error": ours["standard error"], "95 % CI low (log)": ours["95 % CI low"],
    "95 % CI high (log)": ours["95 % CI high"], "tau squared": ours["tau squared"], "I squared %": ours["I squared %"],
    "Q": ours["Q"], "risk ratio": c["estimate"], "risk ratio 95 % CI low": c["estimate 95 % CI low"],
    "risk ratio 95 % CI high": c["estimate 95 % CI high"], "prediction interval low": p["estimate 95 % CI low"],
    "prediction interval high": p["estimate 95 % CI high"], "p-value": ours["p-value"]})
dl = combined(combine(bcg, **BCG, method="DerSimonian-Laird", interval_by="normal"))
checks.check_values("BCG trials", "combine_studies (DerSimonian-Laird, normal)", dl)
reg = read_rows_by(quiet(bcg.combine_studies, **BCG, using=["ablat"], interval_by="normal"), "study")
checks.check_values("BCG trials", "combine_studies (using ablat)",
                    {"intercept": reg["intercept"]["estimate"], "slope": reg['per 1 more "ablat"']["estimate"]})

agrees  BCG trials | combine_studies (REML, normal) | log risk ratio: simplydata -0.714532342158, Viechtbauer (2010), printed -0.7145; difference 3.23e-05 (allowed 5e-05)
      Source: Viechtbauer (2010), printed (printed) - https://doi.org/10.18637/jss.v036.i03
agrees  BCG trials | combine_studies (REML, normal) | standard error: simplydata 0.179781516105, Viechtbauer (2010), printed 0.1798; difference 1.85e-05 (allowed 5e-05)
      Source: Viechtbauer (2010), printed (printed) - https://doi.org/10.18637/jss.v036.i03
agrees  BCG trials | combine_studies (REML, normal) | 95 % CI low (log): simplydata -1.06689763881, Viechtbauer (2010), printed -1.0669; difference 2.36e-06 (allowed 5e-05)
      Source: Viechtbauer (2010), printed (printed) - https://doi.org/10.18637/jss.v036.i03
agrees  BCG trials | combine_studies (REML, normal) | 95 % CI high (log): simplydata -0.362167045506, Viechtbauer (2010), printed -0.3622; difference 3.30e-05 (allowed 5e-05)
      Source: Viechtbauer (2010), pr

agrees  BCG trials | combine_studies (using ablat) | intercept: simplydata 0.251468209979, Viechtbauer (2010), printed 0.2515; difference 3.18e-05 (allowed 5e-05)
      Source: Viechtbauer (2010), printed (printed) - https://doi.org/10.18637/jss.v036.i03
agrees  BCG trials | combine_studies (using ablat) | slope: simplydata -0.0291017250112, Viechtbauer (2010), printed -0.0291; difference 1.73e-06 (allowed 5e-05)
      Source: Viechtbauer (2010), printed (printed) - https://doi.org/10.18637/jss.v036.i03


2

### Every study's log risk ratio and standard error

Against R `metafor::escalc(measure = "RR")` (closed form, 1e-10).

In [4]:
studies = combine(bcg, **BCG)
each_study("BCG trials", "combine_studies (each study)", studies, "log risk ratio")

agrees  BCG trials | combine_studies (each study) | Aronson 1948: log risk ratio: simplydata -0.88931133392, R metafor escalc -0.88931133392; relative difference 0.00e+00 (allowed 1e-10)
      Source: R metafor escalc (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/escalc.html (version 5.2.1)
agrees  BCG trials | combine_studies (each study) | Aronson 1948: standard error: simplydata 0.570600354893, R metafor escalc 0.570600354893; relative difference 1.95e-16 (allowed 1e-10)
      Source: R metafor escalc (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/escalc.html (version 5.2.1)
agrees  BCG trials | combine_studies (each study) | Ferguson & Simes 1949: log risk ratio: simplydata -1.5853886572, R metafor escalc -1.5853886572; relative difference 0.00e+00 (allowed 1e-10)
      Source: R metafor escalc (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/escalc.html (version 5.2.1)
agrees  BCG trials | combine_studies

### Every `method=` and `interval_by=` against R metafor

The models fitted by iteration (REML, Paule-Mandel) are compared with R metafor run to convergence,
`control = list(tol = 1e-12, threshold = 1e-12)`: metafor's defaults stop at a tolerance of 1e-5 (the root search of
Paule-Mandel, the change in tau-squared of REML). REML with the normal interval was checked above (its printed values
and its p-value).

In [5]:
RUNS = [("REML, Hartung-Knapp", {}),
        ("DerSimonian-Laird, Hartung-Knapp", {"method": "DerSimonian-Laird"}),
        ("Paule-Mandel, normal", {"method": "Paule-Mandel", "interval_by": "normal"}),
        ("Paule-Mandel, Hartung-Knapp", {"method": "Paule-Mandel"}),
        ("common effect", {"method": "common effect"})]
for label, options in RUNS:
    checks.check_values("BCG trials", f"combine_studies ({label})", combined(combine(bcg, **BCG, **options)))

agrees  BCG trials | combine_studies (REML, Hartung-Knapp) | estimate: simplydata -0.714532342158, R metafor rma (tol 1e-12) -0.714532342158; relative difference 3.93e-14 (allowed 1e-06)
      Source: R metafor rma (tol 1e-12) (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  BCG trials | combine_studies (REML, Hartung-Knapp) | 95 % CI low: simplydata -1.10844371369, R metafor rma (tol 1e-12) -1.10844371369; relative difference 3.81e-14 (allowed 1e-06)
      Source: R metafor rma (tol 1e-12) (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  BCG trials | combine_studies (REML, Hartung-Knapp) | 95 % CI high: simplydata -0.320620970631, R metafor rma (tol 1e-12) -0.320620970631; relative difference 4.36e-14 (allowed 1e-06)
      Source: R metafor rma (tol 1e-12) (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  

agrees  BCG trials | combine_studies (DerSimonian-Laird, Hartung-Knapp) | estimate: simplydata -0.714117222073, R metafor rma -0.714117222073; relative difference 0.00e+00 (allowed 1e-10)
      Source: R metafor rma (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  BCG trials | combine_studies (DerSimonian-Laird, Hartung-Knapp) | 95 % CI low: simplydata -1.10782135417, R metafor rma -1.10782135417; relative difference 0.00e+00 (allowed 1e-10)
      Source: R metafor rma (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  BCG trials | combine_studies (DerSimonian-Laird, Hartung-Knapp) | 95 % CI high: simplydata -0.320413089978, R metafor rma -0.320413089978; relative difference 3.46e-16 (allowed 1e-10)
      Source: R metafor rma (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  BCG trials | combine_studies (Der

agrees  BCG trials | combine_studies (Paule-Mandel, normal) | estimate: simplydata -0.714968153492, R metafor rma (tol 1e-12) -0.714968153493; relative difference 2.39e-12 (allowed 1e-06)
      Source: R metafor rma (tol 1e-12) (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  BCG trials | combine_studies (Paule-Mandel, normal) | 95 % CI low: simplydata -1.06951033398, R metafor rma (tol 1e-12) -1.06951033399; relative difference 9.63e-12 (allowed 1e-06)
      Source: R metafor rma (tol 1e-12) (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  BCG trials | combine_studies (Paule-Mandel, normal) | 95 % CI high: simplydata -0.360425973002, R metafor rma (tol 1e-12) -0.360425972995; relative difference 1.91e-11 (allowed 1e-06)
      Source: R metafor rma (tol 1e-12) (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agree

agrees  BCG trials | combine_studies (common effect) | estimate: simplydata -0.430285163654, R metafor rma -0.430285163654; relative difference 0.00e+00 (allowed 1e-10)
      Source: R metafor rma (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  BCG trials | combine_studies (common effect) | 95 % CI low: simplydata -0.509661258426, R metafor rma -0.509661258426; relative difference 2.18e-16 (allowed 1e-10)
      Source: R metafor rma (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  BCG trials | combine_studies (common effect) | 95 % CI high: simplydata -0.350909068882, R metafor rma -0.350909068882; relative difference 0.00e+00 (allowed 1e-10)
      Source: R metafor rma (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  BCG trials | combine_studies (common effect) | standard error: simplydata 0.040498751710

### Mantel-Haenszel and Peto, and odds ratios and risk differences from counts

Mantel-Haenszel risk and odds ratios against R `metafor::rma.mh`; Peto's odds ratio against R `metafor::rma.peto`;
REML with Hartung-Knapp on odds ratios (`show="odds ratios"`) and risk differences against R `rma`.

In [6]:
for show in ("risk ratios", "odds ratios"):
    checks.check_values("BCG trials", f"combine_studies (Mantel-Haenszel, {show})",
                        combined(combine(bcg, **BCG, method="Mantel-Haenszel", show=show)))
checks.check_values("BCG trials", "combine_studies (Peto)", combined(combine(bcg, **BCG, method="Peto")))
for show, scale in (("odds ratios", "log"), ("risk differences", "same")):
    checks.check_values("BCG trials", f"combine_studies (REML, {show})", combined(combine(bcg, scale, **BCG, show=show)))

agrees  BCG trials | combine_studies (Mantel-Haenszel, risk ratios) | estimate: simplydata -0.453709604684, R metafor rma.mh -0.453709604684; relative difference 1.22e-16 (allowed 1e-10)
      Source: R metafor rma.mh (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.mh.html (version 5.2.1)
agrees  BCG trials | combine_studies (Mantel-Haenszel, risk ratios) | 95 % CI low: simplydata -0.530809426998, R metafor rma.mh -0.530809426998; relative difference 0.00e+00 (allowed 1e-10)
      Source: R metafor rma.mh (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.mh.html (version 5.2.1)
agrees  BCG trials | combine_studies (Mantel-Haenszel, risk ratios) | 95 % CI high: simplydata -0.376609782369, R metafor rma.mh -0.376609782369; relative difference 1.47e-16 (allowed 1e-10)
      Source: R metafor rma.mh (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.mh.html (version 5.2.1)
agrees  BCG trials | combine_studies

agrees  BCG trials | combine_studies (Peto) | estimate: simplydata -0.474446310941, R metafor rma.peto -0.474446310941; relative difference 0.00e+00 (allowed 1e-10)
      Source: R metafor rma.peto (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.peto.html (version 5.2.1)
agrees  BCG trials | combine_studies (Peto) | 95 % CI low: simplydata -0.554136644845, R metafor rma.peto -0.554136644845; relative difference 2.00e-16 (allowed 1e-10)
      Source: R metafor rma.peto (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.peto.html (version 5.2.1)
agrees  BCG trials | combine_studies (Peto) | 95 % CI high: simplydata -0.394755977037, R metafor rma.peto -0.394755977037; relative difference 4.22e-16 (allowed 1e-10)
      Source: R metafor rma.peto (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.peto.html (version 5.2.1)
agrees  BCG trials | combine_studies (Peto) | standard error: simplydata 0.0406590807446, 

explained difference – precision limit (C29)  BCG trials | combine_studies (REML, risk differences) | estimate: simplydata -0.0251732881463, R metafor rma -0.0251735000367; relative difference 8.42e-06 (allowed 1e-06)
      why: References run with R's default stopping rule; simplydata now converges tighter (its A7 fix) - https://github.com/bioforkgithub/simplydata_tests/blob/main/FINDINGS.md#c29-references-run-with-rs-default-stopping-rule-simplydata-now-converges-tighter-its-a7-fix
      Source: R metafor rma (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  BCG trials | combine_studies (REML, risk differences) | estimate: simplydata -0.0251732881463, R metafor rma (tol 1e-12) -0.0251732881433; relative difference 1.19e-10 (allowed 1e-06)
      Source: R metafor rma (tol 1e-12) (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
explained difference – precision limit (C29)  B

### Subgroups, meta-regression and small-study effects

- `for_each_group_in="alloc"`: each subgroup gets its own REML fit (with Hartung-Knapp), and the test of their
  difference is Q between the subgroups' estimates with their normal-interval standard errors (common-effect
  meta-regression on the groups), as simplydata states it does (and as RevMan and R meta do). R: one `rma` per
  subgroup, and `rma(b, sei = se, mods = ~ group, method = "EE")`.
- `using=["ablat"]`: R `rma(yi, vi, mods = ~ ablat, test = "adhoc")`.
- Peters' test (risk ratios from counts): R meta `metabias(method.bias = "Peters")` (metafor has no Peters' test).
- Trim and fill: R `metafor::trimfill` on the REML, Hartung-Knapp model.

In [7]:
sub = combine(bcg, **BCG, for_each_group_in="alloc")
difference = row_of(sub, "subgroup difference")
values = {"Q between": difference["Q"], "p between": difference["p-value"]}
for group in ("alternate", "random", "systematic"):
    s = row_of(sub, "combined in subgroup", group)
    values[f"{group}: log risk ratio"] = s["estimate on its scale"]
    values[f"{group}: tau squared"] = s["tau squared"]
    values[f"{group}: 95 % CI low (log)"] = s["estimate 95 % CI low on its scale"]
checks.check_values("BCG trials", "combine_studies (for_each_group_in alloc)", values)

reg = read_rows_by(quiet(bcg.combine_studies, **BCG, using=["ablat"]), "study")
values = {}
for term in ("intercept", 'per 1 more "ablat"'):
    for quantity, column in (("estimate", "estimate"), ("95 % CI low", "estimate 95 % CI low"),
                             ("95 % CI high", "estimate 95 % CI high"), ("p-value", "p-value")):
        values[f"{term}: {quantity}"] = reg[term][column]
checks.check_values("BCG trials", "combine_studies (using ablat)", values)

peters, fill = row_of(studies, "small-study effects"), row_of(studies, "trim and fill")
checks.check_values("BCG trials", "combine_studies (Peters' test)", {"t": peters["estimate"], "p-value": peters["p-value"]})
checks.check_values("BCG trials", "combine_studies (trim and fill)",
                    {"studies added": fill["studies added"], "log risk ratio": fill["estimate on its scale"],
                     "95 % CI low (log)": fill["estimate 95 % CI low on its scale"]})

agrees  BCG trials | combine_studies (for_each_group_in alloc) | Q between: simplydata 1.86144494616, R metafor rma 1.86144614123; relative difference 6.42e-07 (allowed 1e-06)
      Source: R metafor rma (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  BCG trials | combine_studies (for_each_group_in alloc) | Q between: simplydata 1.86144494616, R metafor rma (tol 1e-12) 1.86144494615; relative difference 4.59e-12 (allowed 1e-06)
      Source: R metafor rma (tol 1e-12) (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  BCG trials | combine_studies (for_each_group_in alloc) | p between: simplydata 0.394268758884, R metafor rma 0.394268523295; relative difference 5.98e-07 (allowed 1e-06)
      Source: R metafor rma (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  BCG trials | combine_studies (for_each_group_in 

3

## Passive smoking: odds ratios given with their intervals (`estimates=`)

37 studies of lung cancer in non-smoking women married to smokers. Hackshaw et al. (1997) published the pooled excess
risk 24 % (95 % CI 13 % to 36 %), the DerSimonian-Laird odds ratio 1.24 (1.13 to 1.36) with a normal interval
(tolerance half a unit in the second decimal). R: `metafor::conv.wald` (log odds ratio and standard error from the
interval), then `rma` for what is not printed; Egger's test from `metafor::regtest(model = "lm")`; trim and fill from
`metafor::trimfill`.

In [8]:
smoke = load("smoke")
SMOKE = dict(names="name", estimates="odds ratio")
dl = combine(smoke, **SMOKE, method="DerSimonian-Laird", interval_by="normal")
c = row_of(dl, "combined")
values = combined(dl)
values.update({"odds ratio": c["estimate"], "95 % CI low": c["estimate 95 % CI low"],
               "95 % CI high": c["estimate 95 % CI high"]})
checks.check_values("passive smoking", "combine_studies (DerSimonian-Laird, normal)", values)
default = combine(smoke, **SMOKE)
checks.check_values("passive smoking", "combine_studies (REML, Hartung-Knapp)", combined(default))
egger, fill = row_of(default, "small-study effects"), row_of(default, "trim and fill")
checks.check_values("passive smoking", "combine_studies (Egger's test)", {"t": egger["estimate"], "p-value": egger["p-value"]})
checks.check_values("passive smoking", "combine_studies (trim and fill)",
                    {"studies added": fill["studies added"], "log odds ratio": fill["estimate on its scale"]})

agrees  passive smoking | combine_studies (DerSimonian-Laird, normal) | 95 % CI low: simplydata 1.12922353518, Hackshaw et al. (1997), printed 1.13; difference 7.76e-04 (allowed 5e-03)
      Source: Hackshaw et al. (1997), printed (printed) - https://doi.org/10.1136/bmj.315.7114.980
agrees  passive smoking | combine_studies (DerSimonian-Laird, normal) | 95 % CI high: simplydata 1.35831860068, Hackshaw et al. (1997), printed 1.36; difference 1.68e-03 (allowed 5e-03)
      Source: Hackshaw et al. (1997), printed (printed) - https://doi.org/10.1136/bmj.315.7114.980
agrees  passive smoking | combine_studies (DerSimonian-Laird, normal) | standard error: simplydata 0.0471226394375, R metafor rma 0.0471226394375; relative difference 0.00e+00 (allowed 1e-10)
      Source: R metafor rma (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  passive smoking | combine_studies (DerSimonian-Laird, normal) | p-value: simplydata 5.65257581618e-06

2

## Averages, events in time, and correlations

- **Normand 1999** (9 stroke-unit trials, length of stay): Hedges' g (`show="standardised differences"`, the default)
  and the difference in days (`show="differences in averages"`); R `escalc(measure = "SMD")` and `"MD"`.
- **Curtis 1998** (102 CO2 experiments): the log response ratio, R `escalc(measure = "ROM")`.
- **Hart 1999** (6 warfarin trials): strokes per patient-year, R `escalc(measure = "IRR")`.
- **McDaniel 1994** (160 studies): correlations through Fisher's z, R `escalc(measure = "ZCOR")` (compared on the z
  scale, where the model is fitted; simplydata shows r).

All with simplydata's defaults (REML, Hartung-Knapp) against R `rma(test = "adhoc")` (1e-6), and each study's
estimate and standard error against `escalc` (1e-10).

In [9]:
AVERAGES = dict(averages=["m1i", "m2i"], standard_deviations=["sd1i", "sd2i"], counts=["n1i", "n2i"])
normand = load("normand")
SETS = [  # dataset label, table, method label, scale the model is fitted on, simplydata options
    ("Normand 1999", normand, "default", "same", dict(names="source", **AVERAGES)),
    ("Normand 1999", normand, "differences in averages", "same",
     dict(names="source", **AVERAGES, show="differences in averages")),
    ("Curtis 1998", load("curtis"), "ratios of averages", "log", dict(names="name", **AVERAGES, show="ratios of averages")),
    ("Hart 1999", load("hart"), "default", "log", dict(names="study", events=["x1i", "x2i"], time=["t1i", "t2i"])),
    ("McDaniel 1994", load("mcdaniel"), "correlations", "z", dict(names="name", correlations="ri", counts="ni")),
]
normand.combine_studies(names="source", **AVERAGES)

Note from R (metafor): One or more SMD values are unusually large (i.e., abs(yi) > 2). Maybe SEs were used in place of SDs to compute them?.
Combining 9 studies (random effects, REML, Hartung-Knapp): standardised difference (Hedges' g) of "m1i" against "m2i".
Combined standardised difference (Hedges' g) -0.537 (95 % CI -1.25 to 0.176, Hartung-Knapp; 9 studies).
If there were truly no effect on average (a standardised difference (Hedges' g) of 0), a combined effect this far from 0 or further would turn up in about 12 of 100 sets of studies like these (p = 0.121). This does not show that there is no effect.
In a new setting like these studies, the true standardised difference (Hedges' g) would lie between -2.71 and 1.63 (95 % prediction interval, t with 8 degrees of freedom, as metafor uses with Hartung-Knapp; Higgins et al. 2009 use 7).
The true effects typically differ from the average by tau = 0.889 (in standardised differences: the typical spread of the true effects; tau-squared = 0.

,study,row kind,measure,estimate,estimate 95 % CI low,estimate 95 % CI high,standard error,weight %,combined on,p-value,studies,tau,tau squared,I squared %,Q,Q degrees of freedom,Q p-value
1,Edinburgh,study,standardised difference (Hedges' g),-0.355170,-0.579195,-0.131144,0.114301,11.851103,plain scale,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Orpington-Mild,study,standardised difference (Hedges' g),-0.347940,-0.845589,0.149709,0.253907,11.138852,plain scale,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Orpington-Moderate,study,standardised difference (Hedges' g),-2.317569,-2.737075,-1.898063,0.214038,11.387240,plain scale,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Orpington-Severe,study,standardised difference (Hedges' g),-1.887982,-2.673480,-1.102485,0.400771,10.013227,plain scale,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Montreal-Home,study,standardised difference (Hedges' g),-0.383964,-1.272313,0.504385,0.453247,9.562802,plain scale,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Montreal-Transfer,study,standardised difference (Hedges' g),0.172149,-0.204402,0.548699,0.192121,11.509697,plain scale,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Newcastle,study,standardised difference (Hedges' g),0.272052,-0.209107,0.753211,0.245494,11.193842,plain scale,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Umea,study,standardised difference (Hedges' g),-0.424596,-0.663543,-0.185649,0.121914,11.824651,plain scale,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Uppsala,study,standardised difference (Hedges' g),0.289556,-0.083722,0.662834,0.190451,11.518586,plain scale,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,combined,combined,standardised difference (Hedges' g),-0.537108,-1.250322,0.176106,0.309286,100.000000,plain scale,0.120665,9.0,0.889293,0.790843,95.494937,123.729274,8.0,5.622513e-23


In [10]:
for label, table, show, scale, options in SETS:
    rows = combine(table, scale, **options)
    checks.check_values(label, f"combine_studies ({show})", combined(rows))
    each_study(label, f"combine_studies ({show}) (each study)", rows, "estimate")

agrees  Normand 1999 | combine_studies (default) | estimate: simplydata -0.537108258433, R metafor rma -0.537108258567; relative difference 2.49e-10 (allowed 1e-06)
      Source: R metafor rma (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  Normand 1999 | combine_studies (default) | estimate: simplydata -0.537108258433, R metafor rma (tol 1e-12) -0.537108258433; relative difference 2.91e-14 (allowed 1e-06)
      Source: R metafor rma (tol 1e-12) (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  Normand 1999 | combine_studies (default) | 95 % CI low: simplydata -1.25032220907, R metafor rma -1.25032220923; relative difference 1.29e-10 (allowed 1e-06)
      Source: R metafor rma (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  Normand 1999 | combine_studies (default) | 95 % CI low: simplydata -1.25032220907,

agrees  Normand 1999 | combine_studies (differences in averages) | estimate: simplydata -15.1060274735, R metafor rma -15.1060274744; relative difference 6.04e-11 (allowed 1e-06)
      Source: R metafor rma (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  Normand 1999 | combine_studies (differences in averages) | estimate: simplydata -15.1060274735, R metafor rma (tol 1e-12) -15.1060274735; relative difference 1.18e-16 (allowed 1e-06)
      Source: R metafor rma (tol 1e-12) (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  Normand 1999 | combine_studies (differences in averages) | 95 % CI low: simplydata -36.3187490826, R metafor rma -36.3187490843; relative difference 4.76e-11 (allowed 1e-06)
      Source: R metafor rma (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  Normand 1999 | combine_studies (differ

explained difference – precision limit (C29)  Curtis 1998 | combine_studies (ratios of averages) | estimate: simplydata 0.25396167517, R metafor rma 0.253962265876; relative difference 2.33e-06 (allowed 1e-06)
      why: References run with R's default stopping rule; simplydata now converges tighter (its A7 fix) - https://github.com/bioforkgithub/simplydata_tests/blob/main/FINDINGS.md#c29-references-run-with-rs-default-stopping-rule-simplydata-now-converges-tighter-its-a7-fix
      Source: R metafor rma (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  Curtis 1998 | combine_studies (ratios of averages) | estimate: simplydata 0.25396167517, R metafor rma (tol 1e-12) 0.253961675165; relative difference 1.65e-11 (allowed 1e-06)
      Source: R metafor rma (tol 1e-12) (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
explained difference – precision limit (C29)  Curtis 1998 | com

agrees  Hart 1999 | combine_studies (default) | estimate: simplydata -1.01681571481, R metafor rma -1.01681571481; relative difference 0.00e+00 (allowed 1e-06)
      Source: R metafor rma (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  Hart 1999 | combine_studies (default) | estimate: simplydata -1.01681571481, R metafor rma (tol 1e-12) -1.01681571481; relative difference 0.00e+00 (allowed 1e-06)
      Source: R metafor rma (tol 1e-12) (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  Hart 1999 | combine_studies (default) | 95 % CI low: simplydata -1.43700109393, R metafor rma -1.43700109393; relative difference 0.00e+00 (allowed 1e-06)
      Source: R metafor rma (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  Hart 1999 | combine_studies (default) | 95 % CI low: simplydata -1.43700109393, R metafor rma (

agrees  McDaniel 1994 | combine_studies (correlations) | estimate: simplydata 0.237393455728, R metafor rma 0.237393474101; relative difference 7.74e-08 (allowed 1e-06)
      Source: R metafor rma (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  McDaniel 1994 | combine_studies (correlations) | estimate: simplydata 0.237393455728, R metafor rma (tol 1e-12) 0.237393455728; relative difference 1.60e-13 (allowed 1e-06)
      Source: R metafor rma (tol 1e-12) (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  McDaniel 1994 | combine_studies (correlations) | 95 % CI low: simplydata 0.202277133281, R metafor rma 0.202277145952; relative difference 6.26e-08 (allowed 1e-06)
      Source: R metafor rma (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html (version 5.2.1)
agrees  McDaniel 1994 | combine_studies (correlations) | 95 % CI low: simply

## Result

In [11]:
checks.finish()

08_combining_studies: 904 comparisons - 866 agrees, 38 explained difference – precision limit
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)


,notebook,dataset,method,quantity,simplydata,reference,reference source,difference,tolerance,tolerance type,score,status,reason,result,finding,explanation,link,kind,url,ran on
0,08_combining_studies,BCG trials,"combine_studies (REML, normal)",log risk ratio,-0.714532,-0.714500,"Viechtbauer (2010), printed",-3.234216e-05,5.000000e-05,absolute,3.234216e-05,PASS,,agrees,,,,printed,https://doi.org/10.18637/jss.v036.i03,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
1,08_combining_studies,BCG trials,"combine_studies (REML, normal)",standard error,0.179782,0.179800,"Viechtbauer (2010), printed",-1.848389e-05,5.000000e-05,absolute,1.848389e-05,PASS,,agrees,,,,printed,https://doi.org/10.18637/jss.v036.i03,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
2,08_combining_studies,BCG trials,"combine_studies (REML, normal)",95 % CI low (log),-1.066898,-1.066900,"Viechtbauer (2010), printed",2.361189e-06,5.000000e-05,absolute,2.361189e-06,PASS,,agrees,,,,printed,https://doi.org/10.18637/jss.v036.i03,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
3,08_combining_studies,BCG trials,"combine_studies (REML, normal)",95 % CI high (log),-0.362167,-0.362200,"Viechtbauer (2010), printed",3.295449e-05,5.000000e-05,absolute,3.295449e-05,PASS,,agrees,,,,printed,https://doi.org/10.18637/jss.v036.i03,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
4,08_combining_studies,BCG trials,"combine_studies (REML, normal)",tau squared,0.313243,0.313200,"Viechtbauer (2010), printed",4.325814e-05,5.000000e-05,absolute,4.325814e-05,PASS,,agrees,,,,printed,https://doi.org/10.18637/jss.v036.i03,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
899,08_combining_studies,McDaniel 1994,combine_studies (correlations) (each study),study 158: standard error,0.045129,0.045129,R metafor escalc,2.081668e-17,1.000000e-10,relative,4.612669e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/meta...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
900,08_combining_studies,McDaniel 1994,combine_studies (correlations) (each study),study 159: estimate,0.234189,0.234189,R metafor escalc,0.000000e+00,1.000000e-10,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/meta...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
901,08_combining_studies,McDaniel 1994,combine_studies (correlations) (each study),study 159: standard error,0.101015,0.101015,R metafor escalc,6.938894e-17,1.000000e-10,relative,6.869155e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/meta...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
902,08_combining_studies,McDaniel 1994,combine_studies (correlations) (each study),study 160: estimate,0.309520,0.309520,R metafor escalc,1.110223e-16,1.000000e-10,relative,3.586923e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/meta...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
